# Fine Tuning Open AI Models

This notebook is based on the current guidance provided in the [Fine Tuning](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst) documentation from Open AI.

> **Note:** This notebook's example output was generated against `gpt-3.5-turbo`, which is now retired for both inference and fine-tuning on Azure OpenAI in Microsoft Foundry (and deprecated by OpenAI directly). The walkthrough and concepts below are still accurate, but if you're starting a new fine-tuning job today, target a currently supported model instead - for example `gpt-4o-mini` or `gpt-4.1-mini`. See the [Fine-tuning models list](https://learn.microsoft.com/en-us/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst#fine-tuning-models) for the current set of fine-tunable models.

Fine-tuning improves the performance of foundation models for your application by retraining it with additional data and context relevant to that specific use case or scenario. Note that prompt engineering techniques like _few shot learning_ and _retrieval augmented generation_ allow you to enhance the default prompt with relevant data to improve quality. However, these approaches are limited by max token window size of the targeted foundation model. 

With fine-tuning, we are effectively retraining the model itself with the required data (allowing us to use many more examples than can fit in the max token window) - and deploying a _custom_ version of the model that no longer needs to have examples provided at inference time. This not only improves the effectivenenss of our prompt design (we have more flexibility in using the token window for other things) but potentially also improves our costs (by reducing the number of tokens we need to send to the model at inference time).

Fine tuning has 4 steps:
1. Prepare the training data and upload it.
1. Run the training job to get a fine-tuned model.
1. Evaluate the fine-tuned model and iterate for quality.
1. Deploy the fine-tuned model for inference when satisfied.

Note that not all foundation models support fine-tuning - [check OpenAI documentation](https://platform.openai.com/docs/guides/fine-tuning/what-models-can-be-fine-tuned?WT.mc_id=academic-105485-koreyst) for the latest information. You can also fine-tune a previously fine-tuned model. In this tutorial, we'll use `gpt-35-turbo` as our target foundation model for fine-tuning (see the note above for a current, supported alternative). 

---


### 步驟 1.1：準備您的數據集

讓我們建立一個聊天機器人，通過用打油詩回答元素的問題，幫助你理解元素週期表。在_這個_簡單的教學中，我們只會創建一個數據集，用幾個範例回應來訓練模型，這些回應展示了數據的預期格式。在實際應用中，您需要創建一個包含更多範例的數據集。如果存在適合您應用領域的開放數據集，您也可以將其重新格式化以用於微調。

由於我們著重於`gpt-35-turbo`並尋求單輪回應（聊天完成），我們可以使用[這個建議格式](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset?WT.mc_id=academic-105485-koreyst)來創建範例，該格式符合 OpenAI 聊天完成的要求。如果您期待多輪對話內容，您將使用[多輪範例格式](https://platform.openai.com/docs/guides/fine-tuning/multi-turn-chat-examples?WT.mc_id=academic-105485-koreyst)，其中包含一個`weight`參數，用來標示哪些消息應當在微調過程中使用（或不使用）。

我們在本教學中將使用較簡單的單輪格式。數據採用[jsonl格式](https://jsonlines.org/?WT.mc_id=academic-105485-koreyst)，每行一個紀錄，每個紀錄表示為 JSON 格式的物件。以下程式片段顯示了兩個紀錄作為範例—完整範例集（10個範例）請參閱[training-data.jsonl](../../../../../18-fine-tuning/python/openai/training-data.jsonl)，這是我們微調教學將使用的資料集。**注意：** 每筆紀錄_必須_定義於單行內（不可如典型格式化 JSON 檔案那樣跨行）。

```json
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Gallium"}, {"role": "assistant", "content": "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}]}
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Hydrogen"}, {"role": "assistant", "content": "Hydrogen, the first in the line - The lightest of all, so divine - It's in water, you see - And in stars, it's the key - The universe's most common sign"}]}
```

在實際應用中，您需要更多範例集以獲得較佳結果—權衡點在於回應品質與微調所需時間/成本。我們使用小範例集是為了快速完成微調，便於說明流程。詳見[此 OpenAI Cookbook 範例](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst)以獲取更複雜的微調教學。


---

### 第 1.2 步 上傳您的數據集

使用檔案 API 上傳數據，[如此處所述](https://platform.openai.com/docs/guides/fine-tuning/upload-a-training-file)。請注意，要運行此代碼，您必須先完成以下步驟：
 - 安裝 `openai` Python 套件（確保使用版本 >=0.28.0 以獲得最新功能）
 - 將 `OPENAI_API_KEY` 環境變數設置為您的 OpenAI API 金鑰
想了解更多，請參閱課程提供的[設定指南](./../../../00-course-setup/02-setup-local.md?WT.mc_id=academic-105485-koreyst)。

現在，運行代碼從您的本地 JSONL 文件創建一個要上傳的檔案。


In [24]:
from openai import OpenAI
client = OpenAI()

ft_file = client.files.create(
  file=open("./training-data.jsonl", "rb"),
  purpose="fine-tune"
)

print(ft_file)
print("Training File ID: " + ft_file.id)

FileObject(id='file-JdAJcagdOTG6ACNlFWzuzmyV', bytes=4021, created_at=1715566183, filename='training-data.jsonl', object='file', purpose='fine-tune', status='processed', status_details=None)
Training File ID: file-JdAJcagdOTG6ACNlFWzuzmyV


---

### 第 2.1 步：使用 SDK 建立微調作業


In [25]:
from openai import OpenAI
client = OpenAI()

ft_filejob = client.fine_tuning.jobs.create(
  training_file=ft_file.id, 
  model="gpt-3.5-turbo"
)

print(ft_filejob)
print("Fine-tuning Job ID: " + ft_filejob.id)

FineTuningJob(id='ftjob-Usfb9RjasncaZ5Cjbuh1XSCh', created_at=1715566184, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(n_epochs='auto', batch_size='auto', learning_rate_multiplier='auto'), model='gpt-3.5-turbo-0125', object='fine_tuning.job', organization_id='org-EZ6ag0n0S6Zm8eV9BSWKmE6l', result_files=[], seed=830529052, status='validating_files', trained_tokens=None, training_file='file-JdAJcagdOTG6ACNlFWzuzmyV', validation_file=None, estimated_finish=None, integrations=[], user_provided_suffix=None)
Fine-tuning Job ID: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh


---

### 步驟 2.2：檢查工作的狀態

以下是您可以使用 `client.fine_tuning.jobs` API 做的一些事情：
- `client.fine_tuning.jobs.list(limit=<n>)` - 列出最後 n 筆微調工作
- `client.fine_tuning.jobs.retrieve(<job_id>)` - 取得特定微調工作的詳細資料
- `client.fine_tuning.jobs.cancel(<job_id>)` - 取消微調工作
- `client.fine_tuning.jobs.list_events(fine_tuning_job_id=<job_id>, limit=<b>)` - 列出該工作的最多 n 個事件
- `client.fine_tuning.jobs.create(model="gpt-35-turbo", training_file="your-training-file.jsonl", ...)`

這個流程的第一步是 _驗證訓練檔案_，確保資料格式正確。


In [26]:
from openai import OpenAI
client = OpenAI()

# List 10 fine-tuning jobs
client.fine_tuning.jobs.list(limit=10)

# Retrieve the state of a fine-tune
client.fine_tuning.jobs.retrieve(ft_filejob.id)

# List up to 10 events from a fine-tuning job
client.fine_tuning.jobs.list_events(fine_tuning_job_id=ft_filejob.id, limit=10)

SyncCursorPage[FineTuningJobEvent](data=[FineTuningJobEvent(id='ftevent-GkWiDgZmOsuv4q5cSTEGscY6', created_at=1715566184, level='info', message='Validating training file: file-JdAJcagdOTG6ACNlFWzuzmyV', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-3899xdVTO3LN7Q7LkKLMJUnb', created_at=1715566184, level='info', message='Created fine-tuning job: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh', object='fine_tuning.job.event', data={}, type='message')], object='list', has_more=False)

In [30]:
# Once the training data is validated
# Track the job status to see if it is running and when it is complete
from openai import OpenAI
client = OpenAI()

response = client.fine_tuning.jobs.retrieve(ft_filejob.id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)

Job ID: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh
Status: running
Trained Tokens: None


---

### 第 2.3 步：追蹤事件以監察進度


In [44]:
# You can also track progress in a more granular way by checking for events
# Refresh this code till you get the `The job has successfully completed` message
response = client.fine_tuning.jobs.list_events(ft_filejob.id)

events = response.data
events.reverse()

for event in events:
    print(event.message)

Step 85/100: training loss=0.14
Step 86/100: training loss=0.00
Step 87/100: training loss=0.00
Step 88/100: training loss=0.07
Step 89/100: training loss=0.00
Step 90/100: training loss=0.00
Step 91/100: training loss=0.00
Step 92/100: training loss=0.00
Step 93/100: training loss=0.00
Step 94/100: training loss=0.00
Step 95/100: training loss=0.08
Step 96/100: training loss=0.05
Step 97/100: training loss=0.00
Step 98/100: training loss=0.00
Step 99/100: training loss=0.00
Step 100/100: training loss=0.00
Checkpoint created at step 80 with Snapshot ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWyyF2:ckpt-step-80
Checkpoint created at step 90 with Snapshot ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWyzhK:ckpt-step-90
New fine-tuned model created: ft:gpt-3.5-turbo-0125:bitnbot::9OFWzNjz
The job has successfully completed


### 第 2.4 步：在 OpenAI 控制台查看狀態


你亦可透過造訪 OpenAI 網站並瀏覽平台上的 _微調_ 部分來查看狀態。這將顯示你目前工作的狀態，並讓你追蹤先前工作執行的歷史。在這張截圖中，你可以看到先前的執行失敗，而第二次執行成功。作為背景，這次情況是因為第一次執行時使用了格式錯誤的 JSON 檔案記錄——修正後，第二次執行成功完成，並使模型可供使用。

![Fine-tuning job status](../../../../../translated_images/zh-MO/fine-tuned-model-status.563271727bf7bfba.webp)


您亦可透過在視覺化儀表板中向下滾動查看更多狀態訊息及指標，如下所示：

| Messages | Metrics |
|:---|:---|
| ![Messages](../../../../../translated_images/zh-MO/fine-tuned-messages-panel.4ed0c2da5ea1313b.webp) |  ![Metrics](../../../../../translated_images/zh-MO/fine-tuned-metrics-panel.700d7e4995a65229.webp)|


---

### 第3.1步：在程式碼中獲取ID並測試微調模型


In [46]:
# Retrieve the identity of the fine-tuned model once ready
response = client.fine_tuning.jobs.retrieve(ft_filejob.id)
fine_tuned_model_id = response.fine_tuned_model
print("Fine-tuned Model ID:", fine_tuned_model_id)

Fine-tuned Model ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWzNjz


In [ ]:
# You can then use that model to generate completions from the SDK as shown
# Or you can load that model into the OpenAI Playground (in the UI) to validate it from there.
from openai import OpenAI
client = OpenAI()

completion = client.responses.create(
  model=fine_tuned_model_id,
  input=[
    {"role": "system", "content": "You are Elle, a factual chatbot that answers questions about elements in the periodic table with a limerick"},
    {"role": "user", "content": "Tell me about Strontium"},
  ],
  store=False,
)
print(completion.output_text)


ChatCompletionMessage(content="Strontium, a metal so bright - It's in fireworks, a dazzling sight - It's in bones, you see - And in tea, it's the key - It's the fortieth, so pure, that's the right", role='assistant', function_call=None, tool_calls=None)


---

### 步驟 3.2：在 Playground 載入及測試微調模型

您現在可以用兩種方式測試微調模型。首先，您可以訪問 Playground，並使用「模型」下拉選單，從列出的選項中選擇剛微調完成的模型。另一個選項是在微調面板中使用「Playground」選項（請參閱上方截圖），該選項會啟動以下的_比較_視圖，能夠並排展示基礎模型與微調模型版本，便於快速評估。

![微調作業狀態](../../../../../translated_images/zh-MO/fine-tuned-playground-compare.56e06f0ad8922016.webp)

簡單填寫您在訓練資料中使用的系統上下文並提供測試問題。您會注意到兩邊都會更新為相同的上下文和問題。執行比較後，您將看到它們輸出之間的差異。_注意微調模型如何按照您範例中提供的格式來回應，而基礎模型僅僅遵循系統提示。_

![微調作業狀態](../../../../../translated_images/zh-MO/fine-tuned-playground-launch.5a26495c983c6350.webp)

您會注意到比較結果還提供了每個模型的標記數量及推理所需時間。<strong>這個特殊例子相當簡單，意在展示流程，並不反映真實世界的數據集或場景。</strong>您可能會發現兩個樣本中的標記數量相同（系統上下文和用戶提示相同），但微調模型的推理時間較長（自訂模型）。

在真實世界的場景中，您不會使用這種玩具範例，而是針對實際數據（例如用於客戶服務的產品目錄）進行微調，屆時回應品質會更明顯。在_那個_情境下，要用基礎模型達到等效的回應品質，將需要更多自訂提示工程，這會增加標記使用量以及可能增加推理所需的處理時間。_想嘗試看看，請先參考 OpenAI Cookbook 中的微調範例開始。_

---


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
